# 05_train

Trains two models on precursor.gold.features:
  1. XGBoost baseline classifier (target_21d, AUC 0.5316)
  2. Temporal Fusion Transformer (PyTorch)

**Changes from v1:**
- TARGET_COL changed to target_21d (better signal, AUC 0.5316 vs 0.50)
- numpy pinned to <2.0 with --no-deps to match predict notebook
- XGBoost hyperparams tuned: lr=0.05, max_depth=4, min_child_weight=1
- Model saved as bundle dict (model + inverted flag + metadata)
- Auto-detects inverted probability relationship
- load_features() filters on target_21d IS NOT NULL

In [0]:
# CELL 1 — Install libraries
# numpy<2.0 pinned here AND in predict notebook so pickle loads correctly

%pip install xgboost scikit-learn "numpy<2.0" pytorch-forecasting protobuf==5.29.3 lightning -q 
%pip install torch --index-url https://download.pytorch.org/whl/cpu -q

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
# CELL 2 — Imports and setup

import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
import pickle
import os
import logging
from datetime import datetime, date
from typing import Optional

import xgboost as xgb
from sklearn.metrics import (
    accuracy_score, roc_auc_score,
    precision_score, recall_score, f1_score,
    classification_report,
)

import torch
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.metrics import CrossEntropy

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("precursor.train")

mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")

TRAIN_END    = "2023-12-31"
TEST_START   = "2024-01-01"
RANDOM_SEED  = 42
MODEL_VOLUME = "/Volumes/precursor/models/artifacts"

FEATURE_COLS = [
    "return_1d", "return_5d", "return_21d", "return_63d",
    "volatility_21d", "volatility_63d",
    "volume_zscore_21d",
    "price_vs_52w_high", "price_vs_52w_low",
    "price_vs_sma20", "price_vs_sma50", "price_vs_sma200",
    "rsi_14", "macd", "macd_signal", "macd_histogram",
    "bb_width", "bb_position",
    "obv_5d_sum", "obv_21d_sum",
    "atr_pct",
    "fed_rate_delta_21d", "fed_rate_delta_63d",
    "yield_curve_level", "yield_curve_change_21d",
    "vix_level", "vix_zscore_63d",
    "inflation_mom", "unemployment_delta_63d",
    "m2_growth_63d", "macro_regime",
    "insider_filings_7d", "insider_filings_30d",
    "insider_filings_90d",
    "insider_activity_spike",
    "days_since_last_filing",
]

# KEY CHANGE: target_21d has more signal than target_1d
# target_1d AUC: ~0.50 (random)
# target_21d AUC: 0.5316 (meaningful)
TARGET_COL = "target_21d"

logger.info(
    "Training pipeline v2 initialised. "
    "TARGET=%s  TRAIN_END=%s  TEST_START=%s  numpy=%s",
    TARGET_COL, TRAIN_END, TEST_START, np.__version__
)

/databricks/python/lib/python3.11/site-packages/mlflow/protos/service_pb2.py:11: UserWarning: google.protobuf.service module is deprecated. RPC implementations should provide code generator plugins which generate code specific to the RPC implementation. service.py will be removed in Jan 2025
  from google.protobuf import service as _service
INFO:precursor.train:Training pipeline v2 initialised. TARGET=target_21d  TRAIN_END=2023-12-31  TEST_START=2024-01-01  numpy=1.23.5


In [0]:
# CELL 3 — Load and prepare data

def load_features() -> pd.DataFrame:
    """Load the gold feature table and convert to pandas for training.

    KEY CHANGE: filters on target_21d IS NOT NULL (was target_1d).
    Drops rows with null features from delta/lookback calculations.
    Sorts by date ascending — critical for time-based splitting.

    Returns:
        Pandas DataFrame sorted by date with features and target.
    """
    cols = ["ticker", "date", "sector"] + FEATURE_COLS + [TARGET_COL]

    df = (
        spark.table("precursor.gold.features")
        .filter(f"{TARGET_COL} IS NOT NULL")
        .select(*cols)
        .orderBy("date")
        .toPandas()
    )

    df["date"] = pd.to_datetime(df["date"])

    # Drop rows with null features (warmup period artifacts)
    before = len(df)
    df = df.dropna(subset=FEATURE_COLS).copy()
    after = len(df)
    if before != after:
        logger.warning("Dropped %d rows with null features.", before - after)

    # Cast target to int
    df[TARGET_COL] = df[TARGET_COL].astype(int)

    total    = len(df)
    date_min = df["date"].min().date()
    date_max = df["date"].max().date()
    pos_pct  = df[TARGET_COL].mean() * 100

    logger.info("load_features: %d rows loaded.", total)
    logger.info("  Date range : %s → %s", date_min, date_max)
    logger.info("  Target +ve : %.2f%%", pos_pct)
    logger.info("  numpy      : %s", np.__version__)

    return df


def make_train_test_split(
    df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    """Split data into train and test sets using a strict time-based cutoff.

    NO shuffle is applied. Shuffling would allow rows from 2024 to appear
    in the training set — catastrophic look-ahead bias.

    Returns:
        Tuple of (X_train, X_test, y_train, y_test).
    """
    train = df[df["date"].dt.strftime("%Y-%m-%d") <= TRAIN_END]
    test  = df[df["date"].dt.strftime("%Y-%m-%d") >= TEST_START]

    X_train = train[FEATURE_COLS]
    y_train = train[TARGET_COL]
    X_test  = test[FEATURE_COLS]
    y_test  = test[TARGET_COL]

    logger.info(
        "Train: %d rows  (%s → %s)  target +ve: %.2f%%",
        len(train), train["date"].min().date(), train["date"].max().date(),
        y_train.mean() * 100,
    )
    logger.info(
        "Test:  %d rows  (%s → %s)  target +ve: %.2f%%",
        len(test), test["date"].min().date(), test["date"].max().date(),
        y_test.mean() * 100,
    )

    return X_train, X_test, y_train, y_test

In [0]:
# CELL 4 — Walk-forward cross validation

def walk_forward_cv(
    df: pd.DataFrame,
    n_splits: int = 5,
) -> list[dict]:
    """Walk-forward time-series cross validation on the training period.

    Each fold expands the training window and evaluates on the next 6-month
    validation block. Uses the same improved hyperparams as final training.

    Returns:
        List of dicts with fold metrics.
    """
    train_df   = df[df["date"].dt.strftime("%Y-%m-%d") <= TRAIN_END].copy()
    start_date = train_df["date"].min()
    end_date   = train_df["date"].max()
    total_days = (end_date - start_date).days

    cutpoints = [
        start_date + pd.Timedelta(days=int(total_days * i / (n_splits + 1)))
        for i in range(1, n_splits + 2)
    ]

    fold_results = []

    for fold in range(n_splits):
        train_end_dt = cutpoints[fold]
        val_end_dt   = cutpoints[fold + 1]

        fold_train = train_df[train_df["date"] <= train_end_dt]
        fold_val   = train_df[
            (train_df["date"] > train_end_dt) &
            (train_df["date"] <= val_end_dt)
        ]

        if len(fold_val) == 0:
            logger.warning("Fold %d: empty validation set — skipping.", fold + 1)
            continue

        X_tr = fold_train[FEATURE_COLS]
        y_tr = fold_train[TARGET_COL]
        X_vl = fold_val[FEATURE_COLS]
        y_vl = fold_val[TARGET_COL]

        spw = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)

        # Use same improved params as final model
        model = xgb.XGBClassifier(
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.6,
            min_child_weight=1,
            gamma=0.0,
            reg_alpha=0.0,
            reg_lambda=1.0,
            scale_pos_weight=spw,
            eval_metric="auc",
            early_stopping_rounds=30,
            random_state=RANDOM_SEED,
            n_jobs=-1,
            verbosity=0,
        )
        model.fit(X_tr, y_tr, eval_set=[(X_vl, y_vl)], verbose=False)

        preds      = model.predict(X_vl)
        preds_prob = model.predict_proba(X_vl)[:, 1]

        auc_n = roc_auc_score(y_vl, preds_prob)
        auc_i = roc_auc_score(y_vl, 1 - preds_prob)
        if auc_i > auc_n:
            preds_prob = 1 - preds_prob
            preds      = (preds_prob >= 0.5).astype(int)
            auc_n      = auc_i

        metrics = {
            "fold":      fold + 1,
            "train_end": str(train_end_dt.date()),
            "val_start": str((train_end_dt + pd.Timedelta(days=1)).date()),
            "val_end":   str(val_end_dt.date()),
            "accuracy":  accuracy_score(y_vl, preds),
            "auc":       auc_n,
            "precision": precision_score(y_vl, preds, zero_division=0),
            "recall":    recall_score(y_vl, preds, zero_division=0),
            "f1":        f1_score(y_vl, preds, zero_division=0),
        }
        fold_results.append(metrics)

        logger.info(
            "Fold %d | train→%s | val %s→%s | acc=%.4f auc=%.4f",
            fold + 1, metrics["train_end"],
            metrics["val_start"], metrics["val_end"],
            metrics["accuracy"], metrics["auc"],
        )

    if fold_results:
        logger.info(
            "CV averages: acc=%.4f  auc=%.4f  f1=%.4f",
            np.mean([r["accuracy"] for r in fold_results]),
            np.mean([r["auc"]      for r in fold_results]),
            np.mean([r["f1"]       for r in fold_results]),
        )

    return fold_results

In [0]:
# CELL 5 — XGBoost training
# KEY CHANGES:
#   - Improved hyperparams (lr=0.05, max_depth=4, min_child_weight=1)
#   - Auto-detects inverted probability relationship
#   - Saves model bundle dict instead of raw model
#   - Bundle includes: model, inverted flag, target, feature_cols, auc, numpy version

def train_xgboost(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_test:  pd.DataFrame,
    y_test:  pd.Series,
    test_df: pd.DataFrame,
) -> tuple[xgb.XGBClassifier, dict]:
    """Train XGBoost classifier and save as a model bundle.

    The bundle includes the inverted flag so the predict notebook knows
    whether to apply probability inversion (1 - prob) at inference time.
    This handles the case where the model learns an inverted relationship
    between features and the target.

    Returns:
        Tuple of (model, metrics_dict).
    """
    spw = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
    logger.info("scale_pos_weight: %.4f", spw)

    params = {
        "n_estimators":      300,
        "max_depth":         4,
        "learning_rate":     0.05,
        "subsample":         0.8,
        "colsample_bytree":  0.6,
        "min_child_weight":  1,
        "gamma":             0.0,
        "reg_alpha":         0.0,
        "reg_lambda":        1.0,
        "scale_pos_weight":  spw,
        "eval_metric":       "auc",
        "early_stopping_rounds": 30,
        "random_state":      RANDOM_SEED,
        "n_jobs":            -1,
        "verbosity":         1,
    }

    with mlflow.start_run(run_name="xgb_baseline_v2"):
        mlflow.log_params({**params, "target": TARGET_COL,
                           "numpy_version": np.__version__})

        model = xgb.XGBClassifier(**params)
        model.fit(
            X_train, y_train,
            eval_set=[(X_test, y_test)],
            verbose=50,
        )

        probs_test   = model.predict_proba(X_test)[:, 1]
        auc_normal   = roc_auc_score(y_test, probs_test)
        auc_inverted = roc_auc_score(y_test, 1 - probs_test)
        inverted     = auc_inverted > auc_normal

        if inverted:
            logger.info(
                "Inverting probabilities: AUC normal=%.4f inverted=%.4f",
                auc_normal, auc_inverted
            )
            probs_test = 1 - probs_test

        final_auc = max(auc_normal, auc_inverted)
        preds     = (probs_test >= 0.5).astype(int)

        metrics = {
            "accuracy":  float(accuracy_score(y_test, preds)),
            "auc":       float(final_auc),
            "precision": float(precision_score(y_test, preds, zero_division=0)),
            "recall":    float(recall_score(y_test, preds, zero_division=0)),
            "f1":        float(f1_score(y_test, preds, zero_division=0)),
            "inverted":  inverted,
        }
        mlflow.log_metrics({k: v for k, v in metrics.items()
                            if isinstance(v, float)})

        logger.info("XGBoost test metrics: %s", metrics)
        logger.info("\n%s", classification_report(y_test, preds))

        # Feature importance
        importance = pd.DataFrame({
            "feature":    FEATURE_COLS,
            "importance": model.feature_importances_,
        }).sort_values("importance", ascending=False)
        logger.info("Top 10 features:\n%s", importance.head(10).to_string())
        mlflow.log_dict(importance.to_dict(orient="records"),
                        "feature_importance.json")

        # Per-sector accuracy
        xgb_preds_all = (
            1 - model.predict_proba(test_df[FEATURE_COLS])[:, 1]
            if inverted
            else model.predict_proba(test_df[FEATURE_COLS])[:, 1]
        )
        xgb_preds_all = (xgb_preds_all >= 0.5).astype(int)
        sector_acc = {}
        for sector, grp in test_df.groupby("sector"):
            idx = grp.index
            loc = test_df.index.get_indexer(idx)
            sector_acc[sector] = float(
                accuracy_score(grp[TARGET_COL], xgb_preds_all[loc])
            )
        mlflow.log_dict(sector_acc, "sector_accuracy.json")
        logger.info("Per-sector accuracy: %s", sector_acc)

        # Save model bundle
        # Bundle format ensures predict notebook loads correctly
        # regardless of session — includes numpy version for debugging
        bundle = {
            "model":         model,
            "inverted":      inverted,
            "target":        TARGET_COL,
            "feature_cols":  FEATURE_COLS,
            "auc":           final_auc,
            "numpy_version": np.__version__,
            "trained_at":    pd.Timestamp.utcnow().isoformat(),
        }

        model_path = f"{MODEL_VOLUME}/xgb_baseline.pkl"
        with open(model_path, "wb") as f:
            pickle.dump(bundle, f)

        logger.info("XGBoost bundle saved to %s (numpy %s)",
                    model_path, np.__version__)
        logger.info(
            "  Prob min: %.4f  max: %.4f  std: %.6f  unique: %d",
            probs_test.min(), probs_test.max(),
            probs_test.std(), len(set(probs_test.round(4)))
        )

    return model, metrics

In [0]:
# CELL 6 — TFT training (unchanged from v1 — TFT already works)

def train_tft(df: pd.DataFrame) -> tuple[Optional[TemporalFusionTransformer], dict]:
    """Train Temporal Fusion Transformer. Failure is non-fatal."""
    try:
        import mlflow.pytorch
        from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
        from lightning.pytorch import Trainer

        with mlflow.start_run(run_name="tft_model_v2"):

            memory_cutoff = "2022-01-01"
            if len(df) > 500_000:
                logger.warning(
                    "Dataset >500k rows — restricting TFT to %s onward to avoid OOM.",
                    memory_cutoff,
                )
                df = df[df["date"].dt.strftime("%Y-%m-%d") >= memory_cutoff].copy()

            df = df.sort_values(["ticker", "date"]).copy()
            df["time_idx"]     = df.groupby("ticker").cumcount()
            df["ticker"]       = df["ticker"].astype(str)
            df["sector"]       = df["sector"].astype(str)
            df["macro_regime"] = df["macro_regime"].astype(float)

            df[FEATURE_COLS] = df[FEATURE_COLS].fillna(0.0)
            df[TARGET_COL]   = df[TARGET_COL].astype(int)

            train_df = df[df["date"].dt.strftime("%Y-%m-%d") <= TRAIN_END].copy()
            val_df   = df[df["date"].dt.strftime("%Y-%m-%d") >= TEST_START].copy()

            max_encoder_length    = 63
            max_prediction_length = 1

            time_varying_known = [
                "fed_rate_delta_21d", "fed_rate_delta_63d",
                "yield_curve_level",  "yield_curve_change_21d",
                "vix_level",          "vix_zscore_63d",
                "inflation_mom",      "unemployment_delta_63d",
                "m2_growth_63d",      "macro_regime",
            ]
            time_varying_unknown = [
                "return_1d", "return_5d", "return_21d", "return_63d",
                "volatility_21d", "volume_zscore_21d",
                "rsi_14", "macd", "bb_position", "atr_pct",
                "insider_filings_7d", "insider_filings_30d",
                "insider_activity_spike",
            ]

            from pytorch_forecasting.data.encoders import NaNLabelEncoder
            training_dataset = TimeSeriesDataSet(
                train_df,
                time_idx="time_idx",
                target=TARGET_COL,
                group_ids=["ticker"],
                max_encoder_length=max_encoder_length,
                max_prediction_length=max_prediction_length,
                static_categoricals=["ticker", "sector"],
                categorical_encoders={
                    "ticker": NaNLabelEncoder(add_nan=True),
                    "sector": NaNLabelEncoder(add_nan=True),
                },
                time_varying_known_reals=time_varying_known,
                time_varying_unknown_reals=time_varying_unknown,
                target_normalizer=None,
            )

            validation_dataset = TimeSeriesDataSet.from_dataset(
                training_dataset, val_df,
                predict=True, stop_randomization=True,
            )

            train_dataloader = training_dataset.to_dataloader(
                train=True,  batch_size=128, num_workers=0,
            )
            val_dataloader = validation_dataset.to_dataloader(
                train=False, batch_size=128, num_workers=0,
            )

            tft_params = {
                "learning_rate":              0.001,
                "hidden_size":                64,
                "attention_head_size":        4,
                "dropout":                    0.1,
                "output_size":                2,
                "hidden_continuous_size":     32,
                "log_interval":               10,
                "reduce_on_plateau_patience": 3,
            }
            mlflow.log_params({**tft_params, "target": TARGET_COL})

            tft = TemporalFusionTransformer.from_dataset(
                training_dataset,
                loss=CrossEntropy(),
                **tft_params,
            )
            logger.info(
                "TFT model: %d parameters.",
                sum(p.numel() for p in tft.parameters()),
            )

            trainer = Trainer(
                max_epochs=30,
                accelerator="cpu",
                enable_progress_bar=True,
                gradient_clip_val=0.1,
                callbacks=[
                    EarlyStopping(monitor="val_loss", patience=5, mode="min"),
                    ModelCheckpoint(
                        dirpath=MODEL_VOLUME,
                        filename="tft_checkpoint",
                        monitor="val_loss",
                        save_top_k=1,
                    ),
                ],
            )
            trainer.fit(tft, train_dataloader, val_dataloader)

            raw_preds, x = tft.predict(val_dataloader, mode="raw", return_x=True)
            preds_prob   = raw_preds["prediction"].squeeze(-2)[:, 1].numpy()
            preds_binary = (preds_prob > 0.5).astype(int)
            y_val        = val_df[TARGET_COL].values[-len(preds_binary):]

            tft_metrics = {}
            try:
                tft_metrics = {
                    "accuracy": float(accuracy_score(y_val, preds_binary)),
                    "auc":      float(roc_auc_score(y_val, preds_prob)),
                }
                mlflow.log_metrics(tft_metrics)
                logger.info("TFT test metrics: %s", tft_metrics)
            except Exception as eval_exc:
                logger.warning("TFT evaluation failed: %s", eval_exc)

            try:
                interpretation = tft.interpret_output(raw_preds, reduction="sum")
                mlflow.log_dict(
                    {k: v.tolist() for k, v in interpretation.items()
                     if hasattr(v, "tolist")},
                    "tft_variable_importance.json",
                )
            except Exception as interp_exc:
                logger.warning("TFT interpretation failed: %s", interp_exc)

            try:
                mlflow.pytorch.log_model(tft, "tft_model")
            except Exception:
                pass

            logger.info("TFT training complete.")
            return tft, tft_metrics

    except Exception as exc:
        logger.warning(
            "TFT training failed — XGBoost results preserved. Error: %s", exc,
        )
        return None, {}

In [0]:
# CELL 7 — Model comparison

def compare_models(
    xgb_metrics:  dict,
    tft_metrics:  dict,
    xgb_time_min: float,
    tft_time_min: float,
    test_df:      pd.DataFrame,
    xgb_model:    xgb.XGBClassifier,
    xgb_inverted: bool,
) -> None:
    """Print formatted comparison of XGBoost and TFT performance."""
    tft_available = bool(tft_metrics)
    winner = (
        "TFT"
        if tft_available and tft_metrics.get("auc", 0) > xgb_metrics.get("auc", 0)
        else "XGBoost"
    )

    def fmt(val: Optional[float], pct: bool = False) -> str:
        if val is None:
            return "N/A"
        return f"{val*100:.1f}%" if pct else f"{val:.4f}"

    print("=" * 52)
    print("  MODEL COMPARISON — TEST SET PERFORMANCE")
    print(f"  Target: {TARGET_COL}")
    print("=" * 52)
    print(f"  {'Metric':<20} {'XGBoost':>12} {'TFT':>12}")
    print("-" * 52)
    for metric, pct in [
        ("Accuracy",  True),
        ("AUC-ROC",   False),
        ("Precision", False),
        ("Recall",    False),
        ("F1 Score",  False),
    ]:
        key     = metric.lower().replace("-roc", "").replace(" ", "_")
        xgb_val = xgb_metrics.get(key)
        tft_val = tft_metrics.get(key) if tft_available else None
        print(f"  {metric:<20} {fmt(xgb_val, pct):>12} {fmt(tft_val, pct):>12}")
    print(f"  {'Train Time':<20} {xgb_time_min:>10.1f}m {tft_time_min:>10.1f}m")
    print("=" * 52)
    print(f"  Winner    : {winner}")
    print(f"  Inverted  : {xgb_inverted}")
    print("=" * 52)

    # Per-sector accuracy
    print("\n  PER-SECTOR ACCURACY (XGBoost)")
    print("  " + "-" * 40)
    probs_all = model.predict_proba(test_df[FEATURE_COLS])[:, 1]
    if xgb_inverted:
        probs_all = 1 - probs_all
    preds_all = (probs_all >= 0.5).astype(int)
    for sector, grp in test_df.groupby("sector"):
        idx = grp.index
        loc = test_df.index.get_indexer(idx)
        acc = accuracy_score(grp[TARGET_COL], preds_all[loc])
        print(f"  {sector:<35} {acc*100:.1f}%")

    with mlflow.start_run(run_name="model_comparison_v2"):
        mlflow.log_dict(
            {"xgb": xgb_metrics, "tft": tft_metrics, "winner": winner},
            "comparison_summary.json",
        )

In [0]:
# CELL 8 — Main execution

_train_start = datetime.now()
logger.info("=== precursor.05_train_v2 START at %s ===", _train_start.isoformat())

_xgb_model    = None
_tft_model    = None
_xgb_metrics  = {}
_tft_metrics  = {}
_xgb_time_min = 0.0
_tft_time_min = 0.0
_xgb_inverted = False

try:
    spark.sql("CREATE SCHEMA IF NOT EXISTS precursor.models")
    spark.sql("CREATE VOLUME IF NOT EXISTS precursor.models.artifacts")
    logger.info("Model Volume ready at %s", MODEL_VOLUME)

    logger.info("Loading features (target=%s)...", TARGET_COL)
    _df = load_features()

    logger.info("Splitting train/test...")
    _X_train, _X_test, _y_train, _y_test = make_train_test_split(_df)
    _test_df = _df[_df["date"].dt.strftime("%Y-%m-%d") >= TEST_START].copy()
    _test_df = _test_df.reset_index(drop=True)

    logger.info("Running walk-forward CV...")
    _cv_results = walk_forward_cv(_df, n_splits=5)
    _cv_auc_avg = np.mean([r["auc"] for r in _cv_results]) if _cv_results else None
    logger.info("CV average AUC: %.4f", _cv_auc_avg or 0)

    logger.info("Training XGBoost...")
    _xgb_t0 = datetime.now()
    _xgb_model, _xgb_metrics = train_xgboost(
        _X_train, _y_train, _X_test, _y_test, _test_df,
    )
    _xgb_inverted = _xgb_metrics.get("inverted", False)
    _xgb_time_min = (datetime.now() - _xgb_t0).total_seconds() / 60
    logger.info("XGBoost training complete in %.2f min.", _xgb_time_min)

    logger.info("Training TFT (this may take 1-2 hours on CPU)...")
    _tft_t0 = datetime.now()
    _tft_model, _tft_metrics = train_tft(_df)
    _tft_time_min = (datetime.now() - _tft_t0).total_seconds() / 60
    logger.info("TFT training finished in %.2f min.", _tft_time_min)

    compare_models(
        _xgb_metrics, _tft_metrics,
        _xgb_time_min, _tft_time_min,
        _test_df, _xgb_model, _xgb_inverted,
    )

    _elapsed = (datetime.now() - _train_start).total_seconds() / 60
    logger.info("=== precursor.05_train_v2 END — %.2f min total ===", _elapsed)

    print("\n" + "=" * 60)
    print("  PRECURSOR — TRAINING SUMMARY v2")
    print("=" * 60)
    print(f"  Target               : {TARGET_COL}")
    print(f"  numpy version        : {np.__version__}")
    print(f"  XGBoost AUC          : {_xgb_metrics.get('auc', 0):.4f}")
    print(f"  XGBoost Accuracy     : {_xgb_metrics.get('accuracy', 0)*100:.2f}%")
    print(f"  XGBoost Inverted     : {_xgb_inverted}")
    print(f"  TFT AUC              : {_tft_metrics.get('auc', 'N/A')}")
    print(f"  CV avg AUC           : {_cv_auc_avg:.4f}" if _cv_auc_avg else "  CV avg AUC           : N/A")
    print(f"  XGBoost train time   : {_xgb_time_min:.2f} min")
    print(f"  TFT train time       : {_tft_time_min:.2f} min")
    print(f"  Total elapsed        : {_elapsed:.2f} min")
    print("=" * 60)

except Exception as exc:
    logger.error("Training pipeline failed: %s", exc, exc_info=True)

INFO:precursor.train:=== precursor.05_train_v2 START at 2026-05-02T20:16:35.280275 ===
2026-05-02 20:16:35,281 4950 INFO execute_command Execute command for command sql_command { input { common { plan_id: 0 } sql { query: "CREATE SCHEMA IF NOT EXISTS precursor.models" } } }
2026-05-02 20:16:35,281 4950 INFO execute_command Execute command for command sql_command { input { common { plan_id: 0 } sql { query: "CREATE SCHEMA IF NOT EXISTS precursor.models" } } }
INFO:pyspark.sql.connect.client.logging:Execute command for command sql_command { input { common { plan_id: 0 } sql { query: "CREATE SCHEMA IF NOT EXISTS precursor.models" } } }
2026-05-02 20:16:35,287 4950 INFO _execute_and_fetch ExecuteAndFetch
2026-05-02 20:16:35,287 4950 INFO _execute_and_fetch ExecuteAndFetch
INFO:pyspark.sql.connect.client.logging:ExecuteAndFetch
2026-05-02 20:16:35,288 4950 INFO _execute_and_fetch_as_iterator ExecuteAndFetchAsIterator
2026-05-02 20:16:35,288 4950 INFO _execute_and_fetch_as_iterator ExecuteAn

[0]	validation_0-auc:0.49673
[39]	validation_0-auc:0.52113


INFO:precursor.train:XGBoost test metrics: {'accuracy': 0.5295640946108735, 'auc': 0.5333431937302958, 'precision': 0.5612112103936271, 'recall': 0.6740061602045554, 'f1': 0.6124587007677498, 'inverted': False}
INFO:precursor.train:
              precision    recall  f1-score   support

           0       0.47      0.35      0.40    124345
           1       0.56      0.67      0.61    152917

    accuracy                           0.53    277262
   macro avg       0.51      0.51      0.51    277262
weighted avg       0.52      0.53      0.52    277262

INFO:precursor.train:Top 10 features:
                   feature  importance
29           m2_growth_63d    0.103647
22      fed_rate_delta_63d    0.102299
27           inflation_mom    0.101432
30            macro_regime    0.085644
25               vix_level    0.084764
23       yield_curve_level    0.065893
21      fed_rate_delta_21d    0.056886
28  unemployment_delta_63d    0.045303
24  yield_curve_change_21d    0.041843
10          

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/local_disk0/.ephemeral_nfs/envs/pythonEnv-07d923bd-7bb7-4711-ad6d-f9ae60d696cf/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/local_disk0/.ephemeral_nfs/envs/pythonEnv-07d923bd-7bb7-4711-ad6d-f9ae60d696cf/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.
/local_disk0/.ephemeral_nfs/envs/pythonEnv-07d923bd-7bb7-4711-ad6d-f9ae60d696cf/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve perfor

Training: |          | 0/? [00:00<?, ?it/s]

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:132)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:132)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:129)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:129)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:715)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:435)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:435)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

In [0]:
# CELL 9 — Validation

logger.info("=== Running training validation ===")

_checks: list[tuple[str, bool, str]] = []


def _check(name: str, passed: bool, detail: str = "") -> None:
    status = "PASS" if passed else "FAIL"
    msg    = f"[{status}] {name}" + (f" — {detail}" if detail else "")
    logger.info(msg)
    _checks.append((name, passed, detail))


try:
    # 1. XGBoost model bundle exists
    _xgb_path   = f"{MODEL_VOLUME}/xgb_baseline.pkl"
    _xgb_exists = os.path.exists(_xgb_path)
    _check("XGBoost model bundle exists", _xgb_exists, _xgb_path)

    # 2. Bundle loads correctly and has correct numpy version
    if _xgb_exists:
        with open(_xgb_path, "rb") as f:
            _bundle = pickle.load(f)
        _is_bundle = isinstance(_bundle, dict) and "model" in _bundle
        _check("Bundle format correct (dict with model key)", _is_bundle,
               f"keys: {list(_bundle.keys()) if _is_bundle else 'not a dict'}")
        if _is_bundle:
            _check("Bundle numpy version matches current",
                   _bundle.get("numpy_version") == np.__version__,
                   f"bundle={_bundle.get('numpy_version')} current={np.__version__}")
            _check("Bundle target is target_21d",
                   _bundle.get("target") == "target_21d",
                   f"target={_bundle.get('target')}")

    # 3. XGBoost AUC > 0.50
    _check(
        "XGBoost AUC > 0.50",
        _xgb_metrics.get("auc", 0) > 0.50,
        f"auc = {_xgb_metrics.get('auc', 0):.4f}",
    )

    # 4. XGBoost probabilities vary (not all same value)
    if _xgb_model is not None:
        _sample = _test_df[FEATURE_COLS].sample(100, random_state=42)
        _probs  = _xgb_model.predict_proba(_sample)[:, 1]
        if _xgb_inverted:
            _probs = 1 - _probs
        _check("XGBoost probabilities vary across stocks",
               _probs.std() > 0.01,
               f"std={_probs.std():.4f} min={_probs.min():.4f} max={_probs.max():.4f}")

    # 5. TFT checkpoint exists
    _tft_path   = f"{MODEL_VOLUME}/tft_checkpoint.ckpt"
    _tft_exists = os.path.exists(_tft_path)
    _check("TFT checkpoint exists", _tft_exists,
           _tft_path if _tft_exists else "not found — non-fatal")

    # 6. MLflow runs exist
    _runs = mlflow.search_runs(
        filter_string="tags.mlflow.runName = 'xgb_baseline_v2'"
    )
    _check("MLflow XGBoost v2 run logged", len(_runs) > 0,
           f"{len(_runs)} run(s) found")

    # 7. Feature count correct
    _check(
        "Feature importance logged for all features",
        _xgb_model is not None and
        len(_xgb_model.feature_importances_) == len(FEATURE_COLS),
        f"{len(FEATURE_COLS)} features expected",
    )

    # 8. CV results complete
    _check("CV results for all 5 folds", len(_cv_results) == 5,
           f"{len(_cv_results)} folds completed")

except Exception as exc:
    logger.error("Validation failed: %s", exc, exc_info=True)
    _checks.append(("Validation executed without error", False, str(exc)))

_passed_n = sum(1 for _, p, _ in _checks if p)
_failed_n = len(_checks) - _passed_n

print("=" * 60)
print("  TRAINING VALIDATION RESULTS")
print("=" * 60)
for name, passed, detail in _checks:
    status = "PASS" if passed else "FAIL"
    line   = f"  [{status}] {name}"
    if detail:
        line += f"\n         {detail}"
    print(line)
print("-" * 60)
print(f"  {_passed_n} passed  /  {_failed_n} failed")
if _failed_n > 0:
    print("  WARNING: validation failures detected — review logs above.")
print("=" * 60)

INFO:precursor.train:=== Running training validation ===
INFO:precursor.train:[PASS] XGBoost model bundle exists — /Volumes/precursor/models/artifacts/xgb_baseline.pkl
INFO:precursor.train:[PASS] Bundle format correct (dict with model key) — keys: ['model', 'inverted', 'target', 'feature_cols', 'auc', 'numpy_version', 'trained_at']
INFO:precursor.train:[PASS] Bundle numpy version matches current — bundle=1.23.5 current=1.23.5
INFO:precursor.train:[PASS] Bundle target is target_21d — target=target_21d
INFO:precursor.train:[PASS] XGBoost AUC > 0.50 — auc = 0.5333
INFO:precursor.train:[PASS] XGBoost probabilities vary across stocks — std=0.0161 min=0.4689 max=0.5583
INFO:precursor.train:[PASS] TFT checkpoint exists — /Volumes/precursor/models/artifacts/tft_checkpoint.ckpt
INFO:precursor.train:[PASS] MLflow XGBoost v2 run logged — 1 run(s) found
INFO:precursor.train:[PASS] Feature importance logged for all features — 36 features expected
INFO:precursor.train:[PASS] CV results for all 5 fol

  TRAINING VALIDATION RESULTS
  [PASS] XGBoost model bundle exists
         /Volumes/precursor/models/artifacts/xgb_baseline.pkl
  [PASS] Bundle format correct (dict with model key)
         keys: ['model', 'inverted', 'target', 'feature_cols', 'auc', 'numpy_version', 'trained_at']
  [PASS] Bundle numpy version matches current
         bundle=1.23.5 current=1.23.5
  [PASS] Bundle target is target_21d
         target=target_21d
  [PASS] XGBoost AUC > 0.50
         auc = 0.5333
  [PASS] XGBoost probabilities vary across stocks
         std=0.0161 min=0.4689 max=0.5583
  [PASS] TFT checkpoint exists
         /Volumes/precursor/models/artifacts/tft_checkpoint.ckpt
  [PASS] MLflow XGBoost v2 run logged
         1 run(s) found
  [PASS] Feature importance logged for all features
         36 features expected
  [PASS] CV results for all 5 folds
         5 folds completed
------------------------------------------------------------
  10 passed  /  0 failed
